In [ ]:
#options: cc200, dosenbach160, aal
p_ROI = "cc200"
p_fold = 10
p_center = "Stanford"
p_mode = "whole"
p_augmentation = True
p_Method = "ASD-DiagNet"

In [2]:
parameter_list = [p_ROI,p_fold,p_center,p_mode,p_augmentation,p_Method]
print("*****List of patameters****")
print("ROI atlas: ",p_ROI)
print("per Center or whole: ",p_mode)
if p_mode == 'percenter':
    print("Center's name: ",p_center)
print("Method's name: ",p_Method)
if p_Method == "ASD-DiagNet":
    print("Augmentation: ",p_augmentation)


*****List of patameters****
ROI atlas:  cc200
per Center or whole:  whole
Method's name:  ASD-DiagNet
Augmentation:  True


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from functools import reduce
from sklearn.impute import SimpleImputer
import time
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch
import pyprind
import sys
import pickle
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import KFold, StratifiedKFold
import torch.optim as optim
from sklearn.metrics import confusion_matrix
from scipy import stats
from sklearn import tree
import functools
import numpy.ma as ma # for masked arrays
import pyprind
import random
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

ModuleNotFoundError: No module named 'pandas'

## Importing the data 

In [4]:
def get_key(filename):
    f_split = filename.split('_')
    if f_split[3] == 'rois':
        key = '_'.join(f_split[0:3]) 
    else:
        key = '_'.join(f_split[0:2])
    return key

In [5]:
data_main_path = '/home/taban/autism/paper_autism/acerta-abide/acerta-abide/data/functionals/cpac/filt_global/rois_'+p_ROI#cc200'#path to time series data
flist = os.listdir(data_main_path)
print(len(flist))

for f in range(len(flist)):
    flist[f] = get_key(flist[f])
    

df_labels = pd.read_csv('/home/taban/autism/paper_autism/acerta-abide/acerta-abide/data/phenotypes/Phenotypic_V1_0b_preprocessed1.csv')#path 

df_labels.DX_GROUP = df_labels.DX_GROUP.map({1: 1, 2:0})
print(len(df_labels))

labels = {}
for row in df_labels.iterrows():
    file_id = row[1]['FILE_ID']
    y_label = row[1]['DX_GROUP']
    if file_id == 'no_filename':
        continue
    assert(file_id not in labels)
    labels[file_id] = y_label

NameError: name 'os' is not defined

### Helper functions for computing correlations

In [ ]:
 # either 0 (Control) or 1 (ASD)
def get_label(filename):
    assert (filename in labels)
    return labels[filename]

# # loads a subject’s ROI fMRI time series, computes the Pearson correlation matrix 
# across regions, and returns the unique pairwise correlations as a 1D feature vector.
def get_corr_data(filename):
    #print(filename)
    for file in os.listdir(data_main_path):
        if file.startswith(filename):
            df = pd.read_csv(os.path.join(data_main_path, file), sep='\t')
            
    with np.errstate(invalid="ignore"):
        corr = np.nan_to_num(np.corrcoef(df.T))
        mask = np.invert(np.tri(corr.shape[0], k=-1, dtype=bool))
        m = ma.masked_where(mask == 1, mask)
        return ma.masked_where(m, corr).compressed()

# loads a subject’s ROI fMRI time series, computes the Pearson correlation matrix across regions and returns it
def get_corr_matrix(filename):
    for file in os.listdir(data_main_path):
        if file.startswith(filename):
            df = pd.read_csv(os.path.join(data_main_path, file), sep='\t')
    with np.errstate(invalid="ignore"):
        corr = np.nan_to_num(np.corrcoef(df.T))
        return corr

def confusion(g_turth,predictions):
    tn, fp, fn, tp = confusion_matrix(g_turth,predictions).ravel()
    accuracy = (tp+tn)/(tp+fp+tn+fn)
    sensitivity = (tp)/(tp+fn) #recall
    specificty = (tn)/(tn+fp)
    return accuracy,sensitivity,specificty

# returns regions is a 1-D array of indices, masked by regnum, selecting the strongest positive and strongest negative correlation features.
def get_regs(samplesnames,regnum):
    datas = []
    # samplesnames is the subject filenames (training samples only) and regnum is how many top and bottom features to select
    for sn in samplesnames:
        datas.append(all_corr[sn][0])
    # datas.shape = (num_subjects, num_features) i.e Rows → subject, Columns → correlation features
    datas = np.array(datas)
    avg=[]
    for ie in range(datas.shape[1]):
        # avg is the average correlation strength of one specific brain connection (correlation feature) across all subjects.
        avg.append(np.mean(datas[:,ie]))

    # Convert the list of average correlation strengths into a 1D NumPy array
    avg=np.array(avg)
    # highs = indices of the strongest positive correlation features
    highs=avg.argsort()[-regnum:][::-1]
    # lows = indices of the strongest negative correlation features
    lows=avg.argsort()[:regnum][::-1]
    # Combines top positive correlation feature indices (highs) and top negative correlation feature indices (lows)
    regions=np.concatenate((highs,lows),axis=0)

    #regions is a 1-D array of indices, masked by regnum, selecting the strongest positive and strongest negative correlation features.
    return regions


## Helper functions for computing correlations

In [ ]:
if not os.path.exists('./correlations_file'+p_ROI+'.pkl'):
    pbar=pyprind.ProgBar(len(flist))
    all_corr = {}
    for f in flist:
      
        lab = get_label(f)
        # all_corr is a dictionary where each key is a subject’s filename 
        # and each value is a tuple containing the subject’s correlation feature vector and label.
        all_corr[f] = (get_corr_data(f), lab)
        pbar.update()

    print('Corr-computations finished')

    pickle.dump(all_corr, open('./correlations_file'+p_ROI+'.pkl', 'wb'))
    print('Saving to file finished')

else:
    all_corr = pickle.load(open('./correlations_file'+p_ROI+'.pkl', 'rb'))

## Computing eigenvalues and eigenvector

In [ ]:
if p_Method=="ASD-DiagNet":
    eig_data = {}
    pbar = pyprind.ProgBar(len(flist))
    for f in flist:  
        d = get_corr_matrix(f)
        eig_vals, eig_vecs = np.linalg.eig(d)
        # eig_vals.shape = (200, eig_vecs.shape = (200, 200)

        # ev is one eigenvector, represents a column in eig_vecs
        for ev in eig_vecs.T:
            # It checks that the eigenvector ev has unit length (norm ≈ 1).
            np.testing.assert_array_almost_equal(1.0, np.linalg.norm(ev))
            
        # It adds up the absolute values of all elements in the 1-D array eig_vals, where each element is an eigenvalue of the correlation matrix.   
        sum_eigvals = np.sum(np.abs(eig_vals))
        
        # Make a list of (eigenvalue, eigenvector, norm_eigval) tuples
        eig_pairs = [(np.abs(eig_vals[i]), eig_vecs[:,i], np.abs(eig_vals[i])/sum_eigvals)
                     for i in range(len(eig_vals))]

        # eig_pairs is a list of 200 tuples, each containing one eigenvalue magnitude, its 200-dimensional eigenvector, and its weight.
        
        # Sort the (eigenvalue, eigenvector) tuples from high to low
        # It sorts eig_pairs in descending order of eigenvalue magnitude, so the most important eigenvectors come first.
        eig_pairs.sort(key=lambda x: x[0], reverse=True)

        eig_data[f] = {'eigvals':np.array([ep[0] for ep in eig_pairs]),
                       'norm-eigvals':np.array([ep[2] for ep in eig_pairs]),
                       'eigvecs':[ep[1] for ep in eig_pairs]}

#   (sorted)      eig_data[f] = {
#   'eigvals':        array(shape=(200,)),
#   'norm-eigvals':   array(shape=(200,)),
#   'eigvecs':        list of 200 arrays, each shape (200,)
# }
        pbar.update()

## Calculating Eros similarity

In [ ]:
# For each subject, adds its normalized eigenvalue weights element by element and returns 
# a single vector (1D) of combined weights for all eigenvectors.
def norm_weights(sub_flist):
    num_dim = len(eig_data[flist[0]]['eigvals'])
    norm_weights = np.zeros(shape=num_dim)
    for f in sub_flist:
        norm_weights += eig_data[f]['norm-eigvals'] 
    return norm_weights

# Computes a weighted similarity between two subjects by 
# summing weighted dot products of their corresponding eigenvectors, and returns a single similarity score.
def cal_similarity(d1, d2, weights, lim=None):
    # d1 is collection of eigen vector of subject 1 and each eigen-vector is is 1D. Same for d2.
    # lim is integer, number of top eigen-vector to use
    # weights is a 1D array containing the element-wise sum of per-subject normalized eigenvalue weights across subjects
    res = 0.0
    if lim is None:
        weights_arr = weights.copy()
    else:
        weights_arr = weights[:lim].copy()
        weights_arr /= np.sum(weights_arr)
    # Note: weights are only normalized when lim is set; otherwise similarity is unnormalized but still comparable (check last one).

    for i,w in enumerate(weights_arr):
        res += w*np.inner(d1[i], d2[i])
    return res

## Defining dataset class

In [2]:
class CC200Dataset(Dataset):
    # CC200Dataset is a PyTorch dataset for cc200 fMRI data (how to load and prepare data of one subject)
    # For the PyTorch dataset class, you need, __init__(), __getitem__() & __len__()

    # pkl_filename: path to saved .pkl file containing precomputed correlation data
    # data: this is "all_corr"
    # sample_list: train_samples with filenames
    # augmentation: use data augmentation or not
    # aug_factor: how much to increase the dataset size. 1 -> no increase, 2 -> double that dataset size 
    # num_neighbs: how many nearest neighbors to consider for augmentation
    # eig_data: is eig_data
    # similarity_fn: functools.partial(cal_similarity, lim=lim4sim) -> function used to compute similarity between two subjects
    # verbose: it controls whether debugging or detailed information is printed, and it is enabled only for the fist cross-validation fold.
    # regs: selected feature indices (top/bottom correlations).
    
    def __init__(self, pkl_filename=None, data=None, samples_list=None, 
                 augmentation=False, aug_factor=1, num_neighbs=5,
                 eig_data=None, similarity_fn=None, verbose=False, regs=None):
        # regs: NumPy array 1-D, It contains indices of selected correlation features.
        self.regs=regs
        if pkl_filename is not None:
            if verbose:
                print ('Loading ..!', end=' ')
            self.data = pickle.load(open(pkl_filename, 'rb'))
        elif data is not None:
            self.data = data.copy()
            
        else:
            sys.stderr.write('Eigther PKL file or data is needed!')
            return 

        #if verbose:
        #    print ('Preprocess..!', end='  ')
        if samples_list is None:
            self.flist = [f for f in self.data]
        else:
            self.flist = [f for f in samples_list]
        self.labels = np.array([self.data[f][1] for f in self.flist])
        
        current_flist = np.array(self.flist.copy())

        # NumPy uses a boolean mask to select elements where the condition is True or False, filtering the array element-by-element.
        current_lab0_flist = current_flist[self.labels == 0]
        current_lab1_flist = current_flist[self.labels == 1]
        #if verbose:
        #    print(' Num Positive : ', len(current_lab1_flist), end=' ')
        #    print(' Num Negative : ', len(current_lab0_flist), end=' ')
        
        
        if augmentation:
            self.num_data = aug_factor * len(self.flist)
            self.neighbors = {}
            pbar = pyprind.ProgBar(len(self.flist))
            weights = norm_weights(samples_list)#??
            for f in self.flist:
                label = self.data[f][1]
                # set() converts a list into an unordered collection with no duplicates and fast removal.
                candidates = (set(current_lab0_flist) if label == 0 else set(current_lab1_flist))
                candidates.remove(f)
                # eig_f: np.array(eig_f).shape. = (200, 200) | 200 number of eigen vectors of subject f
                eig_f = eig_data[f]['eigvecs']
                sim_list = []
                for cand in candidates:
                    # eig_cand: np.array(eig_f).shape. = (200, 200) | 200 number of eigen vectors of subject cand
                    eig_cand = eig_data[cand]['eigvecs']
                    # Computes a weighted similarity score between subject f and a candidate subject using their eigenvectors.
                    sim = similarity_fn(eig_f, eig_cand,weights)
                    sim_list.append((sim, cand))
                # Sorts the list of (similarity, subject) pairs in descending order so the most similar subjects come first.
                sim_list.sort(key=lambda x: x[0], reverse=True)
                # item[1] is used because the second element of each tuple is the subject ID, and we only want to store the neighbor names.
                # self.neighbors['sub1'] = ['sub5', 'sub8', 'sub3']
                self.neighbors[f] = [item[1] for item in sim_list[:num_neighbs]]#list(candidates)#[item[1] for item in sim_list[:num_neighbs]]
        
        else:
            self.num_data = len(self.flist)

    # It returns one training example: (features, label), PyTorch calls this function every time it needs a sample.
    def __getitem__(self, index):
        # index is the position of the sample that PyTorch is requesting from the dataset.
        if index < len(self.flist):
            # False: We are in the real part of the dataset.
            
            # file name of the subject
            fname = self.flist[index]
            # 1D NumPy array of the coorelation feature of the subject
            data = self.data[fname][0].copy() #get_corr_data(fname, mode=cal_mode)
            # 1D NumPy array containing only the selected correlation features (indexed by self.regs)
            data = data[self.regs].copy()
            label = (self.labels[index],)
            # Converts data (NumPy array) → PyTorch tensor Converts label → PyTorch tensor, Returns them as a pair
            return torch.FloatTensor(data), torch.FloatTensor(label)

            
        else:
            # False: We are in the augmented part of the dataset.

            # Selects a real subject by cycling through self.flist using modulo 
            #(e.g., if index=105 and len=100 → 105%100=5 → picks subject at position 5).
            f1 = self.flist[index % len(self.flist)]
            # d1 is the 1D NumPy array of correlation features for subject f1, and y1 is its label (0 or 1).
            d1, y1 = self.data[f1][0], self.data[f1][1]
            # Selects only the chosen feature indices (self.regs) from d1, reducing it to the masked 1D correlation feature vector.
            d1=d1[self.regs]
            # Randomly selects one neighbor of subject f1 from its precomputed nearest neighbors for augmentation.
            f2 = np.random.choice(self.neighbors[f1])
            d2, y2 = self.data[f2][0], self.data[f2][1]
            d2=d2[self.regs]
            # Ensures both subjects have the same label before mixing them for augmentation.
            assert y1 == y2
            # Generates a random number between 0 and 1 to control how much each subject contributes when mixing features.
            r = np.random.uniform(low=0, high=1)
            label = (y1,)
            # Creates a synthetic feature vector (1D) by linearly mixing two same-class subjects using random weight r.
            data = r*d1 + (1-r)*d2
            return torch.FloatTensor(data), torch.FloatTensor(label)

            # Why does __getitem__ return (data, label)? Because in supervised learning you always train on: Input features → Target label

    # __len__ returns the total number of samples in the dataset (including augmented ones if enabled).
    def __len__(self):
        return self.num_data

NameError: name 'Dataset' is not defined

## Defining data loader function

In [ ]:
# batch size: Number of samples processed together in one training step.
# num_workers: Number of parallel CPU processes used to load data (higher = faster loading).
# mode: 'train' or 'test'; controls shuffling and whether augmentation is enabled.
#regions: 1D array of selected feature indices used to mask correlation features.
# others same as CC200Dataset
def get_loader(pkl_filename=None, data=None, samples_list=None,
               batch_size=64, 
               num_workers=1, mode='train',
               *, augmentation=False, aug_factor=1, num_neighbs=5,
                 eig_data=None, similarity_fn=None, verbose=False,regions=None):
    """Build and return data loader."""

    # During training: We want the model to see data in random order
    # Turn off augmentation in test: Augmentation creates fake synthetic data, Test set must contain real unseen data
    if mode == 'train':
        shuffle = True
    else:
        shuffle = False
        augmentation=False

    # Creates a CC200Dataset object with the selected subjects, feature mask, and optional augmentation settings.
    dataset = CC200Dataset(pkl_filename=pkl_filename, data=data, samples_list=samples_list,
                           augmentation=augmentation, aug_factor=aug_factor, 
                           eig_data=eig_data, similarity_fn=similarity_fn, verbose=verbose,regs=regions)

    # # Wraps the dataset in a PyTorch DataLoader to load data in batches, optionally shuffle it, and use parallel workers.
    data_loader = DataLoader(dataset,
                             batch_size=batch_size,
                             shuffle=shuffle,
                             num_workers=num_workers)
  
    return data_loader
    # The data is fully prepared and ready, but it is only processed when the DataLoader starts iterating.

## Defining Autoencoder class

In [3]:
# num_inputs: Size of input feature vector
# num_latent: Size of compressed representation, Bottleneck layer size
# tied: If True, decoder weights are tied to encoder weights
# num_classes: Number of output classes
# use_dropout: If True, applies dropout regularization, Helps prevent overfitting. (Randomly turning off (setting to zero) some neurons during training.)

# This creates a PyTorch neural network model.
class MTAutoEncoder(nn.Module):
    #initializing the model
    def __init__(self, num_inputs=990, 
                 num_latent=200, tied=True,
                 num_classes=2, use_dropout=False):
        super(MTAutoEncoder, self).__init__()
        self.tied = tied
        self.num_latent = num_latent

        # creates the encoder layer: Input: 990, Output: 200, Learns weight matrix: 200 * 990, Bias vector of length 200, y=Wx+b
        self.fc_encoder = nn.Linear(num_inputs, num_latent)

        # If tied=False, create a separate decoder:
        if not tied:
            self.fc_decoder = nn.Linear(num_latent, num_inputs)
            
         
        #self.fc_encoder = nn.Linear(num_inputs, num_latent)

        # It defines the classifier layer of the model.
        
        if use_dropout:
            # It’s the classification head of the model: the part that predicts ASD vs Control from the latent features.
            self.classifier = nn.Sequential (
                # nn.Sequential is a PyTorch container that runs layers in order.
                nn.Dropout(p=0.5), #    During training, 50% of latent neurons are randomly turned off
                nn.Linear(self.num_latent, 1), # This is a fully-connected layer that maps: Input size = self.num_latent (e.g., 200), Output size = 1
                # It outputs one logit for binary classification: If logit is large positive → model leans toward class 1 If logit is large negative → model leans toward class 0
            )
        else:
            self.classifier = nn.Sequential (
                nn.Linear(self.num_latent, 1),
            )
            
    # This defines how input flows through the network.
    # x  is batch of correlation features
    def forward(self, x, eval_classifier=False):

        #---encoder----
        # 990 → 200
        x = self.fc_encoder(x) # x.shape = (batch_size, 990) -> x.shape = (batch_size, 200)

        # tanh applies non-linearity, now x is the latent representation.
        x = torch.tanh(x)
        #---encoder----

        #---classification----
        # eval_classifier is a flag (True/False switch) that controls whether the classifier is used during the forward pass.
        if eval_classifier:
            # if true: The model encodes input, computes classification output, also reconstructs input
            # Pass latent vector to classifier
            x_logit = self.classifier(x) # x_logit.shape = (batch_size, 1)
        else:
            # if false: The model only does encoding + reconstruction but No classification output
            x_logit = None

            # x_logit=Wx+b, x_logit.shape = (Batch size, 1), 
            # It can be any real number: [[ 2.3 ], [-1.5 ], [ 0.7 ], [-0.2 ]], Positive → leaning toward class 1 (ASD), Negative → leaning toward class 0 (Control), Near 0 → uncertain
         #---classification----

        #---decoder----
        if self.tied:
            # if true, this uses the transpose of encoder weights as decoder weights. 
            # Encoder Weight shape: (200 × 990), Transpose: (990 × 200), 200 → 990, this reconstructs the input back to original feature size.
            x = F.linear(x, self.fc_encoder.weight.t())
        else:
            # Uses a separate decoder layer:
            # 200 → 990, this reconstructs the input back to original feature size, Decoder learns independently
            x = self.fc_decoder(x)
        #---decoder----
        return x, x_logit # (reconstructed_x, classification_logit)
        
        # During training, the model jointly minimizes reconstruction loss using MSE between 
        # the reconstructed and original input, and classification loss using BCEWithLogitsLoss between the predicted logit and the true label.

        
mtae = MTAutoEncoder()

mtae

NameError: name 'nn' is not defined

## Defining training and testing functions

In [ ]:
# model: MTAutoEncoder
# epoch: current training epoch number (One full pass through the entire training dataset)
# Epoch 1 → model sees all 1000 samples once | Epoch 2 → sees them again
# train_loader: The PyTorch DataLoader that provides batches of: (batch_x, batch_y)
# p_bernoulli: It is a probability used to randomly drop input features.
# mode: Controls what to train: 'ae' → train only autoencoder (reconstruction) 'clf' → train only classifier 'both' → train both together
# lam_factor: Weight for classification loss when using both losses, Total Loss=AE Loss+λ⋅Classification Loss
def train(model, epoch, train_loader, p_bernoulli=None, mode='both', lam_factor=1.0):
    # It tells PyTorch: “The model is now in training mode.”, Dropout randomly turns off neurons, Uses batch statistics (e.g., batch mean & variance).
    model.train()
    train_losses = []
    # batch_x → batch of feature tensors (correlation) | batch_y → batch of labels | i → batch index (0, 1, 2, ...)
    # batch_x.shape = (64, 990) | batch_y.shape = (64, 1), 64 → batch size (number of subjects in one batch), 990 → number of selected correlation features per subject (1D feature vector length)
    for i,(batch_x,batch_y) in enumerate(train_loader):
        # # Skips the batch if its size is not equal to the defined batch_size.
        if len(batch_x) != batch_size:
            continue
        if p_bernoulli is not None:
            # Creates a probability tensor (same shape as batch_x) filled with p_bernoulli for generating the Bernoulli noise mask.
            if i == 0:
                p_tensor = torch.ones_like(batch_x).to(device)*p_bernoulli
            # Generates a random 0/1 mask from p_tensor, where each element is 1 with probability p_bernoulli and 0 otherwise.
            rand_bernoulli = torch.bernoulli(p_tensor).to(device)
            # The purpose is: To randomly drop input features during training so the model learns to reconstruct the original clean input from noisy data.

        # Moves the input data and labels to the specified device (CPU or GPU) for computation.
        data, target = batch_x.to(device), batch_y.to(device)
        # Clears (resets) all previously accumulated gradients before computing new ones.
        optimizer.zero_grad()

        # Compute autoencoder (reconstruction) loss if we are training AE or both tasks (AE and clf).
        if mode in ['both', 'ae']:
            if p_bernoulli is not None:
                # Input is corrupted:data * rand_bernoulli
                # Model reconstructs: rec_noisy
                # False tells the model to skip the classification branch and only perform reconstruction.(def forward(self, x, eval_classifier=False):, eval_classifier = False)
                rec_noisy, _ = model(data*rand_bernoulli, False)
                # This computes the average reconstruction error per sample in the batch.
                loss_ae = criterion_ae(rec_noisy, data) / len(batch_x)
            else:
                # Input is clean, Reconstruction target is clean, Standard autoencoder loss
                rec, _ = model(data, False)
                loss_ae = criterion_ae(rec, data) / len(batch_x)
                # The underscore _ means: “I don’t care about this value.”
                # model returns: (reconstructed_output, classification_logit) But in this case: We only need reconstruction, We do NOT need classification output

        # Compute classification loss if we are training classifier or both tasks (AE and clf).
        if mode in ['both', 'clf']:
            #True → eval_classifier=True, So the classifier branch is activated
            # rec_clean  → reconstructed, input logits → classification output
            rec_clean, logits = model(data, True)
            # Compute classification loss
            loss_clf = criterion_clf(logits, target)

        # We are training autoencoder and classifier together.
        if mode == 'both':
            # Total Loss=Reconstruction Loss+λ⋅Classification Loss
            loss_total = loss_ae + lam_factor*loss_clf # this is for training, backpropagation
            # This stores: [AE_loss, CLF_loss] for that batch.
            # detach() → removes from computation graph, cpu() → moves tensor from GPU to CPU, numpy() → converts to NumPy value | This is just for logging — not for training.
            train_losses.append([loss_ae.detach().cpu().numpy(), 
                                 loss_clf.detach().cpu().numpy()])
            
        # Train only the reconstruction part (ignore classifier).
        elif mode == 'ae':
            # Total Loss=Reconstruction Loss
            loss_total = loss_ae # this is for training, backpropagation
            # This stores: [AE_loss, 0.0], No classification loss was computed | This is just for logging — not for training.
            train_losses.append([loss_ae.detach().cpu().numpy(), 
                                 0.0])

        # Train only the classification branch
        elif mode == 'clf':
            # Total Loss=Classification Loss
            loss_total = loss_clf
            # Since AE is not used, it is recorded as 0.
            train_losses.append([0.0, 
                                 loss_clf.detach().cpu().numpy()])
        # computes gradients of the loss with respect to model parameters for backpropagation.
        loss_total.backward()
        # uses those gradients to update weights.
        optimizer.step()

    # It returns a list containing the losses recorded for each batch during training, Track training progress
    return train_losses       

# model: MTAutoencoder
# criterion: loss function used for measuring reconstruction error during testing.
# test_loader: PyTorch DataLoader containing test data (batch_x, batch_y)
# eval_classifier: If False → only evaluate reconstruction, If True → also evaluate classification performance
# num_batch, If set: Limits number of batches evaluated,  If None: Evaluate all batches

def test(model, criterion, test_loader, 
         eval_classifier=False, num_batch=None):
    # test_loss: accumulate the total reconstruction loss over all test batches
    # n_test: counts total number of test samples processed
    # correct: counts the number of correct classification predictions
    test_loss, n_test, correct = 0.0, 0, 0
    # store all predicted class labels from the test set
    all_predss=[]
    
    if eval_classifier:
        # y_true → to store true labels
        # y_pred → to store predicted probabilities
        y_true, y_pred = [], []
    # "with" is used for context management: Do something temporarily inside this block, and automatically clean up afterward.
    # torch.no_grad(): Temporarily disable gradient tracking inside this block. When the block ends: Gradient tracking automatically returns to normal
    with torch.no_grad():
        # It switches the model to evaluation mode. dropout is disabled. uses stored batch statistics (e.g., means)
        model.eval()
        # It loops over the test dataset batch by batch.
        # batch_x → batch of input features (coorelation features), batch_y → batch of labels
        # If batch size = 64: batch_x.shape = (64, 990), batch_y.shape = (64, 1)
        for i,(batch_x,batch_y) in enumerate(test_loader, 1):
            if num_batch is not None:
                # It optionally limits how many batches are evaluated. If num_batch = 5, then only first 4 batches are evaluated.
                if i >= num_batch:
                    continue
            # Moves the input batch to the computation device (GPU or CPU):
            data = batch_x.to(device)
            # Performs a forward pass through the model. rec → reconstructed input, logits → classification scores (if enabled)
            rec, logits = model(data, eval_classifier)
            # criterion(rec, data) This computes the reconstruction loss,
            # overall: This adds the reconstruction loss of the current test batch to the total test loss.
            test_loss += criterion(rec, data).detach().cpu().numpy() 
            # It increases the total count of test samples processed so far.
            n_test += len(batch_x)
            # It checks whether classification evaluation is enabled.
            if eval_classifier: 
                # torch.sigmoid(logits): logits are raw scores from the classifier, Sigmoid converts them into probabilities values are between: 0 and 1
                # .detach(): Removes from gradient tracking
                # .cpu(): Moves tensor from GPU to CPU
                # .numpy(): Converts tensor to NumPy array
                proba = torch.sigmoid(logits).detach().cpu().numpy()
                # It creates a NumPy array of all 1s with: Same shape as proba Integer type (int32)
                preds = np.ones_like(proba, dtype=np.int32)
                # It sets prediction to 0 wherever the predicted probability is less than 0.5.
                preds[proba < 0.5] = 0
                # It adds the current batch’s predicted labels to the overall prediction list (all_predss).
                all_predss.extend(preds)###????
                # It converts the batch of true labels into a NumPy integer array.
                y_arr = np.array(batch_y, dtype=np.int32)
                # It adds the number of correct predictions in the current batch to the total correct count.
                correct += np.sum(preds == y_arr)
                # It adds the true labels from the current batch to the full list of true labels (y_true).
                y_true.extend(y_arr.tolist())
                 # It adds the predicted labels from the current batch to the full list of predicted labels (y_pred).
                y_pred.extend(proba.tolist())

                
        # It computes classification performance metrics using: y_true → all true labels, all_predss → all predicted class labels
        # mlp_acc  → Accuracy mlp_sens → Sensitivity mlp_spef → Specificity
        # This computes accuracy, sensitivity, and specificity from the true and predicted class labels using a confusion matrix.
        mlp_acc,mlp_sens,mlp_spef = confusion(y_true,all_predss)

    # It returns the three classification performance metrics computed during testing:
    # the main training script can: Print results, Store them per fold (cross-validation), Compare performance across epochs, Report final model performance
    return  mlp_acc,mlp_sens,mlp_spef#,correct/n_test




In [ ]:
# This selects GPU if available, otherwise CPU, and sets it as the computation device.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
# This runs only if: Using ASD-DiagNet model Using whole dataset (not per center)
if p_Method == "ASD-DiagNet" and p_mode == "whole":

    # Number of correlation features per subject
    num_corr = len(all_corr[flist[0]][0])
    print("num_corr:  ",num_corr)

    # # It records the current time in seconds.
    # start =time.time()
    # Number of samples processed in one forward/backward pass.
    batch_size = 8
    # learning_rate_ae: Learning rate for autoencoder (encoder) parameters.
    # learning_rate_clf: Learning rate for classifier parameters.
    learning_rate_ae, learning_rate_clf = 0.0001, 0.0001
    # Number of full passes through the training dataset. 25 epochs = sees dataset 25 times, how many times the dataset is trained over
    num_epochs = 25

    # They are NOT using a denoising autoencoder. No random masking, No input corruption, Normal autoencoder
    p_bernoulli = None
    # This controls whether data augmentation is used.
    augmentation = p_augmentation
    # use_dropout applies only to the classifier head, not to the autoencoder layers (in our code).
    use_dropout = False

    # Original training samples = 100, aug_factor = 2, Then: Dataset size becomes 200, Meaning: Each sample produces one augmented sample.
    aug_factor = 2
    # For each subject: Find top 5 most similar subjects (same class)
    num_neighbs = 5
    # Use only top 2 eigenvectors when computing similarity.
    lim4sim = 2
    # This sets the latent dimension of the autoencoder. Compress features to 1/4 of original size.
    n_lat = int(num_corr/4)
    print(n_lat)
     # It records the current time in seconds.
    start= time.time()

    print('p_bernoulli: ', p_bernoulli)
    print('augmentaiton: ', augmentation, 'aug_factor: ', aug_factor, 
          'num_neighbs: ', num_neighbs, 'lim4sim: ', lim4sim)
    print('use_dropout: ', use_dropout, '\n')

    # It creates a similarity function where the number of eigenvectors used (lim) is fixed in advance.
    sim_function = functools.partial(cal_similarity, lim=lim4sim)
    # Creates an empty list to store: (accuracy, sensitivity, specificity) for each fold.
    crossval_res_kol=[]
    # It creates a NumPy array containing the class labels (0 or 1) for all subjects.
    y_arr = np.array([get_label(f) for f in flist])
    # It converts flist (a Python list of filenames (all subjects)) into a NumPy array.
    flist = np.array(flist)
    kk=0 
    # It repeats the entire cross-validation process 10 times (10-fold cross-validation × 10 repetitions)
    for rp in range(10):
        # p_fold = 10 Then: 10-fold cross-validation Each fold uses 90% train, 10% test
        # n_splits = 10 That means: The dataset is divided into 10 equal parts (folds).
        kf = StratifiedKFold(n_splits=p_fold, random_state=1, shuffle=True)
        # It randomly shuffles the order of elements in flist.
        np.random.shuffle(flist)
        # It rebuilds the label array after shuffling flist.
        y_arr = np.array([get_label(f) for f in flist])
        
        # kf.split(flist, y_arr) returns: Indices for training and testing sets for each fold.
        # kk = fold number, train_index, test_index = indices for that fold
        for kk,(train_index, test_index) in enumerate(kf.split(flist, y_arr)):
            # It selects the actual filenames (subjects) for: Training set Testing set Using the indices
            train_samples, test_samples = flist[train_index], flist[test_index]

            # During first fold → print detailed logs (debug info), During remaining folds → stay quiet to avoid too much output
            verbose = (True if (kk == 0) else False)

            # Takes training subject names and number of features to select, and returns indices of the strongest and weakest correlation features.
            regions_inds = get_regs(train_samples,int(num_corr/4))
            # Sets the input feature dimension equal to the number of selected correlation feature indices.
            num_inpp = len(regions_inds)
            # Sets the latent (compressed) dimension to half of the selected input feature size.
            n_lat = int(num_inpp/2)
            # This creates the training DataLoader using selected features and augmentation for the current cross-validation fold.
            train_loader=get_loader(data=all_corr, samples_list=train_samples, 
                                    batch_size=batch_size, mode='train',
                                    augmentation=augmentation, aug_factor=aug_factor, 
                                    num_neighbs=num_neighbs, eig_data=eig_data, similarity_fn=sim_function, 
                                    verbose=verbose,regions=regions_inds)

            # This creates the test DataLoader using the selected features without augmentation or shuffling.
            # In data loader, shuffling means to randomly change the order of samples before forming batches.
            test_loader=get_loader(data=all_corr, samples_list=test_samples, 
                                   batch_size=batch_size, mode='test', augmentation=False, 
                                   verbose=verbose,regions=regions_inds)
            # This initializes a new autoencoder-classifier model with the selected input size and latent dimension for the current fold.
            model = MTAutoEncoder(tied=True, num_inputs=num_inpp, num_latent=n_lat, use_dropout=use_dropout)
            # It moves the entire model (all weights and parameters) to the selected device ("cuda"=GPU).
            model.to(device)
            # This defines the reconstruction loss as the summed squared error between reconstructed and original inputs for autoencoder.
            criterion_ae = nn.MSELoss(reduction='sum')
            # This defines the binary classification loss that applies sigmoid internally and computes cross-entropy for ASD vs Control prediction.
            criterion_clf = nn.BCEWithLogitsLoss()
            # It creates an SGD optimizer that updates: Encoder weights Classifier weights With possibly different learning rates.
            # {'params': model.fc_encoder.parameters(), 'lr': learning_rate_ae} -> Applies: learning_rate_ae To encoder weights only
            #  {'params': model.classifier.parameters(), 'lr': learning_rate_clf} -> Applies: learning_rate_clf To classifier head
            optimizer = optim.SGD([{'params': model.fc_encoder.parameters(), 'lr': learning_rate_ae},
                                   {'params': model.classifier.parameters(), 'lr':
                                    learning_rate_clf}],
                                  momentum=0.9)
            # Imagine rolling a ball down a hill: Learning rate → step size, Momentum → how much inertia the ball has

            # num_epochs = 25, So the model trains 25 full passes over the dataset.
            # The model is trained jointly for reconstruction and classification for 20 epochs, then fine-tuned for classification only in the last 5 epochs.
            # In Python, range(start, stop): Includes start, Excludes stop    
        for epoch in range(1, num_epochs+1):
                if epoch <= 20:
                    # For epochs 1–20: Both Train autoencoder (reconstruction) & Train classifier (classification) i.e Joint learning, So total loss: Loss=AE+λ⋅CLF
                    train_losses = train(model, epoch, train_loader, p_bernoulli, mode='both')
                else:
                    # For epochs 21–25: Train classifier only Freeze reconstruction objective Now: Loss=CLF ,This fine-tunes classification performance.
                    train_losses = train(model, epoch, train_loader, p_bernoulli, mode='clf')
            # This: Runs the model on the test set, Computes classification metrics, Returns: (accuracy, sensitivity, specificity), These values are stored in: res_mlp
            res_mlp = test(model, criterion_ae, test_loader, eval_classifier=True)
            print(res_mlp)
            # It stores the test results from the current fold (crossval_res_kol).
            crossval_res_kol.append(res_mlp)
        print("averages:")
        # So it computes: Mean accuracy, Mean sensitivity, Mean specificity
        print(np.mean(np.array(crossval_res_kol),axis = 0))
        # Gets the current time (in seconds).
        finish= time.time()
        # Computes elapsed time: Runtime = finish − start Runtime=finish−start
        print(finish-start)

# in the paper: Classification performance using 10-fold cross-validation on the whole dataset; Note that our proposed approach, ASD-DiagNet (with data augmentation) achieves the highest accuracy among other methods.
# Method, ASD-DiagNet -> Accuracy: 70.3, Sensitivity: 68.3, Specificity: 72.2  
# Method, ASD-DiagNet (no aug)-> Accuracy: 69.4, Sensitivity:69.6, Specificity: 69.2  


In [ ]:
# The model trains and evaluates using data from only one specific center (e.g., Stanford).
if p_Method == "ASD-DiagNet" and p_mode == "percenter":
    num_corr = len(all_corr[flist[0]][0])

    # Extract subject keys from filenames.
    flist = os.listdir(data_main_path)
    for f in range(len(flist)):
        flist[f] = get_key(flist[f])

    # First part before _ is center name. Example: Stanford_005 → Stanford
    centers_dict = {}
    for f in flist:
        key = f.split('_')[0]

        if key not in centers_dict:
            # centers_dict = { 'Stanford': [...], 'NYU': [...], ... }
            centers_dict[key] = []
        # added subject in that particular center key (e.g., 'Stanford': [f1,f2,f3...])
        centers_dict[key].append(f)

    
    # Only subjects from one center are used.
    flist = np.array(centers_dict[p_center])

    
    start =time.time()
    #flist = np.array(sorted(os.listdir(data_main_path)))
    batch_size = 8
    learning_rate_ae, learning_rate_clf = 0.0001, 0.0001
    num_epochs = 25

    p_bernoulli = None
    augmentation = p_augmentation
    use_dropout = False

    aug_factor = 2
    num_neighbs = 5
    lim4sim = 2
    n_lat = int(num_corr/4)


    print('p_bernoulli: ', p_bernoulli)
    print('augmentaiton: ', augmentation, 'aug_factor: ', aug_factor, 
          'num_neighbs: ', num_neighbs, 'lim4sim: ', lim4sim)
    print('use_dropout: ', use_dropout, '\n')


    sim_function = functools.partial(cal_similarity, lim=lim4sim)
    all_rp_res=[]
    y_arr = np.array([get_label(f) for f in flist])

    kk=0 
    crossval_res_kol_kol=[]
    for rp in range(10):
        print("========================")
        crossval_res_kol = []
        start= time.time()
        #-------------------------ISSUE-------------
        # In paper: Classification accuracy using 5-fold cross-validation on individual data centers using our proposed method, ASD-DiagNet (with and without data augmentation), compared with other methods.
        # p_fold = 5 (according to the paper)
        #-------------------------ISSUE-------------
        kf = StratifiedKFold(n_splits=p_fold)
        #np.random.shuffle(flist)
        y_arr = np.array([get_label(f) for f in flist])
        for kk,(train_index, test_index) in enumerate(kf.split(flist, y_arr)):
        
            train_samples, test_samples = flist[train_index], flist[test_index]

            verbose = (True if (kk == 0) else False)

            regions_inds = get_regs(train_samples,int(num_corr/4))
            num_inpp = len(regions_inds)
            n_lat = int(num_inpp/2)
            num_inpp = len(regions_inds)
            train_loader=get_loader(data=all_corr, samples_list=train_samples, 
                                    batch_size=batch_size, mode='train',
                                    augmentation=augmentation, aug_factor=aug_factor, 
                                    num_neighbs=num_neighbs, eig_data=eig_data, similarity_fn=sim_function, 
                                    verbose=verbose,regions=regions_inds)

            test_loader=get_loader(data=all_corr, samples_list=test_samples, 
                                   batch_size=batch_size, mode='test', augmentation=False, 
                                   verbose=verbose,regions=regions_inds)

            model = MTAutoEncoder(tied=True, num_inputs=num_inpp, num_latent=n_lat, use_dropout=use_dropout)
            model.to(device)
            criterion_ae = nn.MSELoss(reduction='sum')
            criterion_clf = nn.BCEWithLogitsLoss()
            optimizer = optim.SGD([{'params': model.fc_encoder.parameters(), 'lr': learning_rate_ae},
                                   {'params': model.classifier.parameters(), 'lr': learning_rate_clf}],
                                  momentum=0.9)

            for epoch in range(1, num_epochs+1):
                if epoch <= 20:
                    train_losses = train(model, epoch, train_loader, p_bernoulli, mode='both')
                else:
                    train_losses = train(model, epoch, train_loader, p_bernoulli, mode='clf')


            res_mlp = test(model, criterion_ae, test_loader, eval_classifier=True)
            #print("fold",kk+1,":",test(model, criterion_ae, test_loader, eval_classifier=True))
            crossval_res_kol.append(res_mlp)
        print("Result of repeat ",rp,":")
        print(np.mean(np.array(crossval_res_kol),axis = 0))
        all_rp_res.append(np.mean(np.array(crossval_res_kol),axis = 0))
        finish= time.time()

        print("Running time:",finish-start)
    # this also computes Avergae result of 10 repeats, but not needed on results of the paper
    print("Avergae result of 10 repeats: ",np.mean(np.array(all_rp_res),axis = 0))

In [ ]:
# (SVM or Random Forest), not ASD-DiagNet. This is easy to interpret --- Remember you need to have correct flist here -- i.e all subjects
if p_Method != "ASD-DiagNet" and p_mode == "whole":

    # If: p_Method == "SVM" → use Support Vector Machine, Otherwise → use Random Forest (100 trees)
    clf = SVC(gamma = 'auto') if  p_Method == 'SVM' else RandomForestClassifier(n_estimators=100)
    #Stores results across 10 repetitions.
    overall_result = []
    
    for rp in range(10):
        
        kf = StratifiedKFold(n_splits=p_fold, random_state=1, shuffle=True)
        # Randomize subject order.
        np.random.shuffle(flist)
        y_arr = np.array([get_label(f) for f in flist])
        res = []
        for kk,(train_index, test_index) in enumerate(kf.split(flist, y_arr)):
            train_samples, test_samples = np.array(flist)[train_index], np.array(flist)[test_index]
            train_data = []
            train_labels = []
            test_data = []
            test_labels = []

            for i in train_samples:
                train_data.append(all_corr[i][0])
                train_labels.append(all_corr[i][1])

            for i in test_samples:
                test_data.append(all_corr[i][0])
                test_labels.append(all_corr[i][1])

            
            clf.fit(train_data,train_labels)
            pr = clf.predict(test_data)
            res.append(confusion(test_labels,pr))
        
        print("repeat: ",rp,np.mean(res, axis=0).tolist())
        overall_result.append(np.mean(res, axis=0).tolist())   
    print("---------------Result of repeating 10 times-------------------")
    print(np.mean(np.array(overall_result), axis=0).tolist())

In [ ]:
# (SVM or Random Forest), not ASD-DiagNet. Also, only percenter (e.g., stanford) This is easy to interpret --- Remember you need to have correct flist here -- i.e subjects only of a center
# You have to run the percenter code for the ASD-Diagnet first to get the correct flist, as the filter if written in that code
random.seed(19)
np.random.seed(19)
if p_Method != "ASD-DiagNet" and p_mode == "percenter":
    
    clf = SVC(gamma = 'auto') if  p_Method == 'SVM' else RandomForestClassifier(n_estimators=100)
    overall_result = []
    for rp in range(10):
        kf = StratifiedKFold(n_splits=p_fold, random_state=1, shuffle=True)
        np.random.shuffle(flist)
        y_arr = np.array([get_label(f) for f in flist])
        res = []
        for kk,(train_index, test_index) in enumerate(kf.split(flist, y_arr)):
            train_samples, test_samples = np.array(flist)[train_index], np.array(flist)[test_index]
            train_data = []
            train_labels = []
            test_data = []
            test_labels = []

            for i in train_samples:
                train_data.append(all_corr[i][0])
                train_labels.append(all_corr[i][1])

            for i in test_samples:
                test_data.append(all_corr[i][0])
                test_labels.append(all_corr[i][1])

            clf.fit(train_data,train_labels)
            pr = clf.predict(test_data)
            res.append(confusion(test_labels,pr))
        
        print("repeat: ",rp,np.mean(res, axis=0).tolist())
        overall_result.append(np.mean(res, axis=0).tolist())   
    print("---------------Result of repeating 10 times for: ",p_center,"-------------------")
    print(np.mean(np.array(overall_result), axis=0).tolist())